# Green Bond Thesis — Data Collection Pipeline
**Title:** Do Politicians' Words Have a Dollar Value?  
**Author:** Nahian Ibnat  
**Supervisor:** Zoltán Csaba Tóth  
**Institution:** Central European University — MA in Economics, Data & Policy  
**Date:** April 2026

---

## Overview
This notebook collects and processes political climate sentiment data for India and Indonesia (2021–2024) using the GDELT Global Knowledge Graph and ClimateBERT.

### Pipeline
1. Install dependencies
2. Configure settings and output directory
3. Query GDELT via Google BigQuery
4. Sample articles for efficient processing
5. Load ClimateBERT model
6. Score articles with ClimateBERT
7. Build daily sentiment series
8. Visualise sentiment timeline
9. Download output files

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

---
## Cell 1 — Install Dependencies

In [1]:
!pip install -q transformers torch pandas requests tqdm google-cloud-bigquery db-dtypes

---
## Cell 2 — Imports and Configuration

In [ ]:
import os
import pandas as pd
import torch
from tqdm.notebook import tqdm

# ── Mount Google Drive ─────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Thesis root on Drive ──────────────────────────────
ROOT = "/content/drive/MyDrive/Thesis"

# ── Output directories ────────────────────────────────
DATA_DIR   = os.path.join(ROOT, "data", "processed", "gdelt")   # CSVs
OUTPUT_DIR = os.path.join(ROOT, "output", "gdelt")              # PNGs
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Study parameters ──────────────────────────────────────
PROJECT_ID  = "project-c7cff37e-dfac-4b4b-bc2"   # Google Cloud project ID
COUNTRIES   = ["India", "Indonesia"]
START_DATE  = 20200101000000   # GDELT format: YYYYMMDDHHMMSS
END_DATE    = 20241231235959

# ── Key political events ──────────────────────────────────
# Used for event markers in the sentiment timeline chart
EVENTS = {
    "2022-11-15": "JETP\n(G20 Bali)",
    "2023-01-25": "India Green\nBond",
}

print(f"Data directory   : {DATA_DIR}")
print(f"Output directory : {OUTPUT_DIR}")
print(f"Countries        : {', '.join(COUNTRIES)}")
print(f"Period           : {START_DATE} → {END_DATE}")
print(f"Key events       : {list(EVENTS.keys())}")

---
## Cell 3 — GDELT Data Collection via Google BigQuery

Queries the GDELT 2.0 Global Knowledge Graph (GKG) public table on BigQuery.

**Why BigQuery?**  
The GDELT DOC API was attempted first but abandoned due to rate limiting (HTTP 429 errors) and unreliable bulk collection. BigQuery provides direct access to the full GDELT dataset with no rate limits.

**Date format note:**  
GDELT stores dates as 14-digit integers (`YYYYMMDDHHMMSS`). Standard `YYYYMMDD` format returns 0 results.

**Country filter:**  
Uses GDELT geocoding identifiers (`India#IN`, `Indonesia#ID`) for precision over plain text matching.

**Theme filter:**  
Applied directly in SQL to avoid pulling millions of irrelevant rows.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT
  DATE,
  SourceCommonName  AS source,
  DocumentIdentifier AS url,
  V2Tone            AS tone,
  Locations         AS locations,
  Themes            AS themes
FROM `gdelt-bq.gdeltv2.gkg`
WHERE DATE BETWEEN 20200101000000 AND 20241231235959
  AND (
    Locations LIKE '%India#IN%'
    OR Locations LIKE '%Indonesia#ID%'
  )
  AND (
    Themes LIKE '%CLIMATE%'
    OR Themes LIKE '%ENV_%'
    OR Themes LIKE '%ENERGY%'
    OR Themes LIKE '%GREEN%'
  )
"""

print("Querying GDELT GKG (2021–2024)...")
print("Estimated runtime: 3–5 minutes\n")

df = client.query(query).to_dataframe()
print(f"Records retrieved : {len(df):,}")
print(f"DATE min          : {df['DATE'].min()}")
print(f"DATE max          : {df['DATE'].max()}")

# Assign country label
df["country"] = df["locations"].apply(
    lambda x: "India" if "India#IN" in str(x) else "Indonesia"
)
print("\nArticles by country:")
print(df["country"].value_counts())

# Save raw output
raw_path = f"{DATA_DIR}/gdelt_raw_articles.csv"
df.to_csv(raw_path, index=False)
print(f"\nSaved → {raw_path}")

---
## Cell 4 — Sample Articles for Efficient ClimateBERT Scoring

The raw GDELT pull contains millions of records. Scoring all of them with ClimateBERT would take many hours on GPU.

**Sampling strategy:** 1,000 articles per country per month, sampled randomly with a fixed seed for reproducibility. This yields approximately 48,000 articles — statistically representative while remaining computationally feasible.

In [ ]:
raw_df = pd.read_csv(f"{DATA_DIR}/gdelt_raw_articles.csv")

# Parse date from 14-digit GDELT timestamp
raw_df["date"] = pd.to_datetime(
    raw_df["DATE"].astype(str).str[:8],
    format="%Y%m%d",
    errors="coerce"
)
raw_df["month"] = raw_df["date"].dt.to_period("M")
raw_df = raw_df.dropna(subset=["date"])

print(f"Before sampling : {len(raw_df):,} articles")
print(f"Date range      : {raw_df['date'].min().date()} → {raw_df['date'].max().date()}")

# Sample 1,000 articles per country per month
sampled_df = (
    raw_df.groupby(["country", "month"])
    .apply(lambda x: x.sample(min(len(x), 1000), random_state=42))
    .reset_index(drop=True)
)

print(f"After sampling  : {len(sampled_df):,} articles")
print(f"Months covered  : {sampled_df['month'].nunique()}")
print("\nSampled articles by country:")
print(sampled_df["country"].value_counts())

# Overwrite raw file with sampled version
sampled_df.to_csv(f"{DATA_DIR}/gdelt_raw_articles.csv", index=False)
print(f"\nSaved sampled articles → {DATA_DIR}/gdelt_raw_articles.csv")

---
## Cell 5 — Load ClimateBERT

Loads the `climatebert/distilroberta-base-climate-sentiment` model from HuggingFace.

**Why ClimateBERT?**  
Unlike general-purpose sentiment models, ClimateBERT is fine-tuned on climate-related disclosures and distinguishes between vague environmental claims and credible policy commitments — a critical distinction for measuring political credibility in green bond markets.

**Output labels:**
- `positive` → +1.0 (ambitious climate commitment)
- `neutral` → 0.0
- `negative` → −1.0 (hostile or regressive climate signal)

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "climatebert/distilroberta-base-climate-sentiment"

print(f"Loading {MODEL_NAME}...")
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device     = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
)

print(f"Device : {'GPU ✓' if device == 0 else 'CPU (slower)'}")
print("ClimateBERT ready.")

---
## Cell 6 — Score Articles with ClimateBERT

Scores each article's `themes` field using ClimateBERT in batches of 64.

**Runtime:** ~15–20 minutes on T4 GPU for ~48,000 articles.

**Note on scoring the `themes` column:**  
GDELT `themes` contains structured topic codes (e.g. `CLIMATE_CHANGE;RENEWABLE_ENERGY`) rather than natural language sentences. The GDELT `V2Tone` score is therefore used as the primary continuous sentiment variable, while ClimateBERT scores serve as a robustness check and for categorical classification.

**Note:** ClimateBERT is applied here for robustness classification.
The primary sentiment variable is GDELT V2Tone (Cell 7), which is
continuous and validated for bulk news analysis. ClimateBERT scores
on GDELT theme codes yielded degenerate results (all neutral) due to
structured topic codes lacking natural language properties required
for transformer-based classification.

In [ ]:
LABEL_MAP  = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
BATCH_SIZE = 64

# Load sampled articles
raw_df = pd.read_csv(f"{DATA_DIR}/gdelt_raw_articles.csv")
print(f"Loaded {len(raw_df):,} articles for scoring")

# Score themes column
texts  = raw_df["themes"].fillna("").tolist()
scores = []

print(f"Scoring {len(texts):,} texts in batches of {BATCH_SIZE}...")
for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = [str(t)[:512] for t in texts[i:i+BATCH_SIZE]]
    try:
        results = classifier(batch)
        scores.extend([LABEL_MAP.get(r["label"].lower(), 0.0) for r in results])
    except Exception as e:
        print(f"  Batch {i//BATCH_SIZE} error: {e}")
        scores.extend([0.0] * len(batch))

raw_df["climatebert_score"] = scores

# Save scored articles
scored_path = f"{DATA_DIR}/gdelt_climatebert_scored.csv"
raw_df.to_csv(scored_path, index=False)
print(f"\nSaved scored articles → {scored_path}")
print("\nClimateBERT score distribution:")
print(raw_df["climatebert_score"].value_counts())

---
## Cell 7 — Build Daily Sentiment Series

Aggregates article-level scores to a daily country-level sentiment series.

This produces the key independent variable **S_{c,t}** used in the regression:

$$\text{Sentiment}_{c,t} \in [-1, 1]$$

The primary variable used in regression is `sentiment_gdelt` (GDELT V2Tone).
`sentiment_climatebert` is retained as a robustness check column but is not
used as the main independent variable.

Two sentiment measures are constructed:
- `sentiment_gdelt` — continuous V2Tone score, primary variable
- `sentiment_climatebert` — mean ClimateBERT score, robustness check

`sentiment_std` serves as a proxy for **Signaling Noise** — days with high variance across articles indicate contradictory political signals.

In [ ]:
raw_df = pd.read_csv(f"{DATA_DIR}/gdelt_climatebert_scored.csv")

# Parse date
raw_df["date"] = pd.to_datetime(
    raw_df["DATE"].astype(str).str[:8],
    format="%Y%m%d",
    errors="coerce"
)
raw_df = raw_df.dropna(subset=["date"])

# Parse V2Tone → normalised sentiment score in [-1, 1]
# V2Tone format: "tone,positive,negative,polarity,activity_ref,self_ref,wordcount"
raw_df["sentiment_gdelt"] = raw_df["tone"].apply(
    lambda x: max(-1.0, min(1.0, float(str(x).split(",")[0]) / 10.0))
    if pd.notna(x) else 0.0
)

print(f"Date range : {raw_df['date'].min().date()} → {raw_df['date'].max().date()}")
print(f"Total rows : {len(raw_df):,}")

# Daily aggregation
daily_df = (
    raw_df.groupby(["country", "date"])
    .agg(
        sentiment_gdelt       = ("sentiment_gdelt",      "mean"),
        sentiment_climatebert = ("climatebert_score",    "mean"),
        sentiment_std         = ("sentiment_gdelt",      "std"),   # Signaling Noise proxy
        article_count         = ("sentiment_gdelt",      "count"),
        pct_positive          = ("sentiment_gdelt",      lambda x: (x > 0).mean()),
        pct_negative          = ("sentiment_gdelt",      lambda x: (x < 0).mean()),
    )
    .reset_index()
    .sort_values(["country", "date"])
    .reset_index(drop=True)
)

daily_path = f"{DATA_DIR}/daily_sentiment_gdelt.csv"
daily_df.to_csv(daily_path, index=False)
print(f"\nSaved daily sentiment → {daily_path}")
print(f"Total country-days : {len(daily_df):,}")
print("\nSummary by country:")
print(daily_df.groupby("country").agg(
    days           = ("date",           "nunique"),
    mean_sentiment = ("sentiment_gdelt", "mean"),
    mean_noise     = ("sentiment_std",   "mean"),
    total_articles = ("article_count",   "sum"),
).round(4).to_string())

print("\nNote: sentiment_gdelt is the primary regression variable.")
print("ClimateBERT scores are retained for robustness checks only.")

---
## Cell 8 — Sentiment Timeline Visualisation

Plots the daily GDELT V2Tone sentiment series for India and Indonesia with a 30-day moving average and key political event markers:
- **November 15, 2022** — Indonesia JETP announcement at G20 Bali
- **January 25, 2023** — India's inaugural sovereign green bond auction

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load daily sentiment
daily_df = pd.read_csv(f"{DATA_DIR}/daily_sentiment_gdelt.csv")
daily_df["date"] = pd.to_datetime(daily_df["date"])

print(f"Date range : {daily_df['date'].min().date()} → {daily_df['date'].max().date()}")
print(daily_df.groupby("country")["date"].nunique().rename("days"))

COLORS = {"India": "#E85D24", "Indonesia": "#1D9E75"}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle(
    "Daily Climate Sentiment (GDELT V2Tone) — 2020–2024",
    fontsize=13, y=1.01
)

for ax, (country, color) in zip(axes, COLORS.items()):
    data = daily_df[daily_df["country"] == country].copy().sort_values("date")
    data["smooth"] = data["sentiment_gdelt"].rolling(30, min_periods=5).mean()

    ax.fill_between(data["date"], data["sentiment_gdelt"],
                    alpha=0.15, color=color)
    ax.plot(data["date"], data["smooth"],
            color=color, linewidth=1.8, label="30-day MA")
    ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")

    # Event markers
    for date_str, label in EVENTS.items():
        ed = pd.Timestamp(date_str)
        ax.axvline(ed, color="black", linewidth=1.0, linestyle=":")
        ax.text(ed, -0.15, label, fontsize=7, ha="center", va="top",
                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                          alpha=0.85, ec="gray", linewidth=0.5))

    ax.set_ylabel(country, fontsize=10, fontweight="bold", color=color)
    ax.set_ylim(-1.1, 1.1)
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.5))
    ax.grid(axis="y", linewidth=0.4, alpha=0.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper right", fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.xlabel("Date")
plt.tight_layout()

chart_path = f"{OUTPUT_DIR}/sentiment_timeline.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {chart_path}")

---
## Cell 9 — Download Output Files

Downloads all output files to your local machine.

**Files for supervisor submission:**
- `gdelt_raw_articles.csv` — raw GDELT articles (sampled)
- `gdelt_climatebert_scored.csv` — articles with ClimateBERT scores
- `daily_sentiment_gdelt.csv` — daily S_{c,t} variable, ready for regression
- `sentiment_timeline.png` — visualisation chart

In [ ]:
from google.colab import files

output_files = [
    f"{DATA_DIR}/gdelt_raw_articles.csv",
    f"{DATA_DIR}/gdelt_climatebert_scored.csv",
    f"{DATA_DIR}/daily_sentiment_gdelt.csv",
    f"{OUTPUT_DIR}/sentiment_timeline.png",
]

for f_path in output_files:
    if os.path.exists(f_path):
        print(f"Downloading: {f_path}")
        files.download(f_path)
    else:
        print(f"[skip] Not found: {f_path}")

print("\nAll files downloaded.")
print("\nSubmit to supervisor:")
print("  - gdelt_raw_articles.csv")
print("  - daily_sentiment_gdelt.csv")
print("  - sentiment_timeline.png")